# Coding Theory Basics

| Difficulty | █████░░░░░ 5/10 |
| :--- | :--- |
| Prerequisites | 05.05 |
| Time | ~65 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/05_Information_Theory/06_coding_theory_basics.ipynb)

---

## 🎯 Learning Objective

Learn source coding, Huffman coding, and the minimum description length principle for representing information efficiently.

---

## 📐 Theory

The source coding theorem (`05.01`) says you can never beat $H(X)$ bits per symbol *on average*
for lossless compression. This notebook asks the constructive question it left open: how do you
actually *build* a code that gets close to that bound?

### Prefix codes and why they matter

A **prefix code** (or "prefix-free code") assigns each symbol a binary codeword such that no
codeword is a prefix of any other. This single property is what makes a stream of concatenated
codewords **uniquely decodable** without needing separators: reading bits left to right, the
moment you match a complete codeword, you know decoding must stop there — no other codeword
could still be "in progress" as a prefix. Every prefix code corresponds to a **binary tree**:
symbols sit at leaves, and the path from root to leaf (recording a $0$ for "go left" and $1$ for
"go right" at each step) is exactly that symbol's codeword. This is why building an optimal
prefix code is really a tree-construction problem.

### Kraft's inequality and optimal codeword lengths

For a prefix code with codeword lengths $\ell_1,\dots,\ell_n$, **Kraft's inequality** requires
$\sum_i 2^{-\ell_i} \le 1$ — you can't have too many short codewords, since every short codeword
"uses up" a large fraction of the tree's leaf capacity. Given a source distribution $P$, the
codeword length that minimizes expected code length subject to this constraint is
$\ell_i = -\log_2 P(x_i)$ — *exactly* the surprisal from `05.01`. This is the deep reason entropy
is the right notion of "expected code length": an optimal code assigns each symbol a length
proportional to its own surprisal, so a symbol's code length is short precisely when it's
common. In general $-\log_2 P(x_i)$ isn't an integer, so no *exact* code achieves this — but
**Huffman coding** gets provably as close as any prefix code can.

### Huffman coding: a greedy, provably optimal algorithm

Huffman's algorithm builds the optimal prefix-code tree bottom-up, greedily:

1. Start with one leaf node per symbol, weighted by its frequency (or probability).
2. Repeatedly take the *two lowest-weight* nodes, merge them into a new internal node whose
   weight is their sum, and reinsert the merged node.
3. Stop when only one node (the root) remains. Each symbol's codeword is the sequence of
   left/right choices from root to its leaf.

Merging the two rarest symbols first pushes them deepest into the tree (longest codewords) while
keeping frequent symbols shallow (shortest codewords) — precisely the "short code for common
symbols" intuition Kraft's inequality demanded. Huffman's algorithm is provably optimal among
all prefix codes and achieves expected code length within 1 bit of the entropy bound:
$H(X) \le L_{\text{Huffman}} < H(X) + 1$.

### Beyond Huffman: arithmetic coding and minimum description length

Huffman coding's one weakness is that it must assign each symbol a *whole number* of bits, which
can waste up to 1 bit per symbol when probabilities are far from powers of $\tfrac12$.
**Arithmetic coding** sidesteps this by encoding an entire *message* as a single fractional
number in $[0,1)$, narrowing the representable interval multiplicatively according to each
symbol's probability — this lets it approach the entropy bound arbitrarily closely, even for
awkward probabilities, at the cost of more intricate bookkeeping than Huffman's simple tree.

Both techniques are concrete instances of the **minimum description length (MDL)** principle: the
best model or explanation for data is the one that minimizes the *total* number of bits needed
to describe both the model and the data given the model. A good compressor is implicitly a good
model of its source's statistics — which is exactly the bridge to how tokenizers work, below.

---

In [ ]:
# Setup
import heapq
from collections import Counter
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Building the Huffman tree for a short string makes the "merge the two rarest symbols first"
rule visible: frequent symbols end up shallow (short codes), rare symbols end up deep (long
codes).

In [ ]:
class HuffmanNode:
    """A tree node: leaves carry a real symbol, internal nodes carry the SUM of their
    children's weights and no symbol. A monotonically increasing _uid breaks heap ties
    deterministically (Python's heapq needs __lt__, and equal-weight nodes are common)."""
    _next_id = 0
    def __init__(self, symbol, weight, left=None, right=None):
        self.symbol, self.weight, self.left, self.right = symbol, weight, left, right
        self.uid = HuffmanNode._next_id
        HuffmanNode._next_id += 1
    def __lt__(self, other):
        return (self.weight, self.uid) < (other.weight, other.uid)
    def is_leaf(self):
        return self.left is None and self.right is None

def build_huffman_tree(frequencies):
    heap = [HuffmanNode(sym, freq) for sym, freq in frequencies.items()]
    heapq.heapify(heap)
    while len(heap) > 1:
        left, right = heapq.heappop(heap), heapq.heappop(heap)   # two RAREST remaining nodes
        heapq.heappush(heap, HuffmanNode(None, left.weight + right.weight, left, right))
    return heap[0]

# A short, classic example -- 'abracadabra' has just 5 distinct symbols with clearly
# different frequencies, small enough that the whole tree fits legibly in one plot
word = "abracadabra"
freqs = Counter(word)
root = build_huffman_tree(freqs)

# Build a networkx graph of the tree for plotting (graphviz's `dot` BINARY isn't installed in
# this environment, so we lay out the tree manually rather than using graphviz_layout)
graph, node_labels, edge_labels = nx.DiGraph(), {}, {}
positions, leaf_x = {}, [0]

def add_to_graph_and_layout(node, depth):
    graph.add_node(node.uid)
    node_labels[node.uid] = f"'{node.symbol}': {node.weight}" if node.is_leaf() else str(node.weight)
    if node.is_leaf():
        x = leaf_x[0]; leaf_x[0] += 1
        positions[node.uid] = (x, -depth)
        return x
    x_left = add_to_graph_and_layout(node.left, depth + 1)
    x_right = add_to_graph_and_layout(node.right, depth + 1)
    graph.add_edge(node.uid, node.left.uid); edge_labels[(node.uid, node.left.uid)] = "0"
    graph.add_edge(node.uid, node.right.uid); edge_labels[(node.uid, node.right.uid)] = "1"
    x = (x_left + x_right) / 2
    positions[node.uid] = (x, -depth)
    return x

add_to_graph_and_layout(root, depth=0)

fig, ax = plt.subplots(figsize=(8, 5))
nx.draw(graph, positions, ax=ax, labels=node_labels, with_labels=True, node_size=1400,
        node_color="#4C72B0", font_color="white", font_size=9, arrows=False)
nx.draw_networkx_edge_labels(graph, positions, edge_labels=edge_labels, ax=ax, font_size=10)
ax.set_title(f"Huffman tree for '{word}'  (leaf label = symbol:frequency)")
plt.tight_layout()
plt.show()

def assign_codewords(node, prefix="", codes=None):
    codes = {} if codes is None else codes
    if node.is_leaf():
        codes[node.symbol] = prefix or "0"
    else:
        assign_codewords(node.left, prefix + "0", codes)
        assign_codewords(node.right, prefix + "1", codes)
    return codes

codewords = assign_codewords(root)
print("Symbol frequencies:", dict(freqs))
print("Huffman codewords:  ", codewords)
print("\nNotice: 'a' (frequency 5, the most common symbol) gets the SHORTEST codeword,")
print("while 'c' and 'd' (frequency 1 each, the rarest) get the LONGEST -- exactly the")
print("'short code for common symbols' principle Kraft's inequality predicts.")

## 🐍 Implementation from Scratch

We build the Huffman tree bottom-up exactly as the algorithm specifies, derive each symbol's
codeword by walking root-to-leaf, then encode and decode a message end-to-end -- confirming
perfect round-trip recovery and checking the $H(X) \le L_{\text{Huffman}} < H(X)+1$ bound.

In [ ]:
def huffman_encode(text, codewords):
    return "".join(codewords[ch] for ch in text)

def huffman_decode(bitstring, root):
    decoded, node = [], root
    for bit in bitstring:
        node = node.left if bit == "0" else node.right
        if node.is_leaf():
            decoded.append(node.symbol)
            node = root
    return "".join(decoded)

# --- Round-trip test on the small example from the visualization ---
encoded_bits = huffman_encode(word, codewords)
decoded_text = huffman_decode(encoded_bits, root)
print(f"Original:  {word!r}")
print(f"Encoded:   {encoded_bits}  ({len(encoded_bits)} bits, vs {len(word) * 8} bits as fixed 8-bit ASCII)")
print(f"Decoded:   {decoded_text!r}")
print(f"Round-trip lossless: {decoded_text == word}")

# --- Verify the entropy bound H(X) <= L_Huffman < H(X) + 1 on a longer, more realistic text ---
def entropy_bits(probs, eps=1e-12):
    probs = np.clip(probs, eps, 1.0)
    return -np.sum(probs * np.log2(probs))

longer_text = ("the quick brown fox jumps over the lazy dog while the cat sleeps near the "
               "warm fireplace and the mice scurry quietly across the wooden floor at night")
freqs_long = Counter(longer_text)
root_long = build_huffman_tree(freqs_long)
codewords_long = assign_codewords(root_long)

n_chars = len(longer_text)
probs_long = np.array([f / n_chars for f in freqs_long.values()])
H_text = entropy_bits(probs_long)
avg_huffman_length = sum(freqs_long[ch] * len(codewords_long[ch]) for ch in freqs_long) / n_chars

print(f"\nLonger text: {n_chars} characters, {len(freqs_long)} distinct symbols")
print(f"  Entropy H(X)                  = {H_text:.4f} bits/char")
print(f"  Average Huffman codeword length = {avg_huffman_length:.4f} bits/char")
print(f"  Bound H(X) <= L_Huffman < H(X)+1 holds: {H_text <= avg_huffman_length < H_text + 1}")

total_huffman_bits = sum(freqs_long[ch] * len(codewords_long[ch]) for ch in freqs_long)
print(f"  Total size: {total_huffman_bits} bits (Huffman) vs {n_chars * 8} bits (ASCII) "
      f"-- a {n_chars * 8 / total_huffman_bits:.2f}x reduction")

# Round-trip check on the longer text too -- correctness must hold regardless of message length
encoded_long = huffman_encode(longer_text, codewords_long)
print(f"  Round-trip lossless on longer text: {huffman_decode(encoded_long, root_long) == longer_text}")

## 🤖 Why This Matters for AI

Every modern LLM tokenizer -- GPT's byte-pair encoding (BPE), SentencePiece, WordPiece -- makes
the same frequency-driven bet Huffman coding does: spend short codes (single tokens) on
frequent patterns, and longer codes (multiple tokens) on rare ones. BPE builds its vocabulary by
iteratively merging the *most frequent adjacent pair* of symbols, growing from individual bytes
up to whole common words -- the mirror image of Huffman's "merge the two rarest nodes" rule, but
driven by the identical underlying idea: let empirical frequency dictate structure. We build a
minimal BPE tokenizer from scratch below and show it allocates short (single-token)
representations to frequent substrings, exactly paralleling the Huffman tree's shallow leaves.

In [ ]:
# A minimal Byte-Pair Encoding (BPE) tokenizer trainer, following Sennrich et al. (2016) --
# repeatedly merge the MOST FREQUENT adjacent symbol pair, exactly mirroring Huffman's "merge
# the two RAREST nodes" but applied to building a VOCABULARY instead of a decoding tree.
def get_pair_counts(word_freqs):
    pair_counts = Counter()
    for word, freq in word_freqs.items():
        symbols = word.split()
        for i in range(len(symbols) - 1):
            pair_counts[(symbols[i], symbols[i + 1])] += freq
    return pair_counts

def apply_merge(pair, word_freqs):
    bigram, merged_token = " ".join(pair), "".join(pair)
    return {word.replace(bigram, merged_token): freq for word, freq in word_freqs.items()}

# A tiny corpus with a skewed word-frequency distribution -- 'low' and 'newest' are common,
# 'lower' and 'widest' are rare, so a good tokenizer should give the common words shorter
# (fewer-token) representations, exactly Huffman's "short code for common symbols" idea
corpus = ["low"] * 5 + ["lower"] * 2 + ["newest"] * 6 + ["widest"] * 3
word_freqs = Counter(corpus)
# Represent each word as space-separated characters plus an end-of-word marker '_'
word_freqs = {" ".join(list(word)) + " _": freq for word, freq in word_freqs.items()}

n_merges = 10
for step in range(n_merges):
    pair_counts = get_pair_counts(word_freqs)
    if not pair_counts:
        break
    most_frequent_pair = max(pair_counts, key=pair_counts.get)
    print(f"merge {step + 1:2d}: {most_frequent_pair} "
          f"(seen {pair_counts[most_frequent_pair]} times) -> new token {''.join(most_frequent_pair)!r}")
    word_freqs = apply_merge(most_frequent_pair, word_freqs)

print("\nFinal tokenization of each word type:")
for original_word in ["low", "lower", "newest", "widest"]:
    tokenized = next(k for k in word_freqs if k.replace(" ", "").replace("_", "") == original_word)
    n_tokens = len(tokenized.split())
    print(f"  {original_word!r:10s} (corpus frequency {corpus.count(original_word)}) -> "
          f"{tokenized.split()}  ({n_tokens} token{'s' if n_tokens != 1 else ''})")

print("\n'low' and 'newest' -- the MOST FREQUENT words -- both collapsed into a SINGLE token.")
print("'lower' and 'widest' -- rarer words -- stayed fragmented into multiple tokens.")
print("This is the exact same principle as Huffman's shallow leaves for frequent symbols:")
print("spend your limited vocabulary budget on short representations for common patterns.")

## 🏋️ Exercises

Six levels, in order: mechanical computation → conceptual understanding → derivation →
implementation → application → open-ended challenge. Solutions/outlines are in the collapsed
`Solution` blocks below each exercise — try before revealing.

### 🔢 COMPUTE
1. By hand, build the Huffman tree for symbol frequencies $\{A: 4, B: 2, C: 1, D: 1\}$ (merge
   the two smallest weights repeatedly, as in `build_huffman_tree`). Write down each symbol's
   codeword, then compute $H(X)$ for this distribution. As a sanity check, confirm Kraft's
   inequality $\sum_i 2^{-\ell_i} \le 1$ holds for your codeword lengths (it should hold with
   *equality*, since this distribution is dyadic — every probability is a power of $\tfrac12$),
   then verify everything with the code above.

<details>
<summary>💡 Solution</summary>

Merging smallest weights first: $C(1)+D(1)\to 2$, then this new node $(2)$ merges with $B(2)\to
4$, then that merges with $A(4)\to 8$. Walking root-to-leaf gives codewords $A{=}0$,
$B{=}10$, $C{=}110$, $D{=}111$ (lengths $1,2,3,3$). The probabilities are
$(\tfrac12,\tfrac14,\tfrac18,\tfrac18)$, giving
$H(X)=-\big(\tfrac12\log_2\tfrac12+\tfrac14\log_2\tfrac14+2\times\tfrac18\log_2\tfrac18\big)=
\tfrac12+\tfrac12+\tfrac34=1.75$ bits — and the average Huffman length is
$\tfrac12(1)+\tfrac14(2)+\tfrac18(3)+\tfrac18(3)=1.75$ bits too, an *exact* match (rather than
just satisfying the $H(X)\le L<H(X)+1$ bound with room to spare), because every $-\log_2 P(x_i)$
here happens to already be an integer. Kraft's sum is
$2^{-1}+2^{-2}+2^{-3}+2^{-3}=0.5+0.25+0.125+0.125=1.0$ exactly, confirming equality.
`build_huffman_tree`/`assign_codewords` on this input reproduce the same codewords.

</details>

### 💭 UNDERSTAND
2. This notebook's bound says $H(X) \le L_{\text{Huffman}} < H(X)+1$. Without computing
   anything, predict: what *shape* of distribution makes this gap close to its maximum of nearly
   1 full bit — a distribution with many roughly-equal-probability symbols, or a distribution
   with one symbol at overwhelming probability and the rest sharing a tiny remainder? Then check
   with a concrete example (e.g. frequencies $\{A: 999, B: 1\}$).

<details>
<summary>💡 Solution</summary>

The one-dominant-symbol case makes the gap large, approaching (but never reaching) 1 full bit.
With frequencies $\{A{:}999, B{:}1\}$, $H(X)\approx 0.0114$ bits (both symbols are nearly
deterministic, so there's almost no genuine uncertainty), yet Huffman coding for *any* 2-symbol
alphabet must assign each symbol at least 1 bit — a prefix code can't do better than 1 bit/symbol
once there are 2 or more distinct codewords needed, however skewed the probabilities are. So the
average Huffman length here is exactly $1.0$ bit, and the gap $1.0-0.0114\approx 0.989$ bits is
close to the theoretical ceiling. Pushing the skew further (e.g. $99999{:}1$) drives $H(X)$ even
closer to $0$ while the Huffman length stays pinned at exactly $1.0$ bit, pushing the gap even
closer to (but still strictly under) 1 full bit. A near-uniform distribution, by contrast, makes
Huffman lengths land close to their ideal $-\log_2 P(x_i)$ values, keeping the gap small.

</details>

### ✏️ DERIVE
3. This notebook states that the optimal (possibly fractional) codeword length is
   $\ell_i = -\log_2 P(x_i)$, achieved when Kraft's inequality holds with equality
   ($\sum_i 2^{-\ell_i}=1$). Show that if every symbol's probability happens to be an exact
   power of $\tfrac12$ (a **dyadic** distribution), then a prefix code using exactly these
   integer lengths achieves average code length *equal* to $H(X)$ — i.e. the Huffman bound's gap
   shrinks from "less than 1 bit" to exactly 0.

<details>
<summary>💡 Solution outline</summary>

If $P(x_i) = 2^{-\ell_i}$ for integers $\ell_i$ (the dyadic assumption), then
$-\log_2 P(x_i) = \ell_i$ exactly — the "ideal" codeword length from Kraft's inequality is
already an integer, so no rounding is needed to build an actual prefix code with these lengths
(and such a code exists, since $\sum_i 2^{-\ell_i} = \sum_i P(x_i) = 1$ satisfies Kraft's
inequality with equality). The expected code length of *any* code using lengths $\ell_i$ is by
definition $L=\sum_i P(x_i)\ell_i$. Substituting $\ell_i=-\log_2 P(x_i)$:
$$L = \sum_i P(x_i)\cdot\big(-\log_2 P(x_i)\big) = -\sum_i P(x_i)\log_2 P(x_i) = H(X)$$
which is exactly the definition of entropy — so $L=H(X)$ precisely, with zero gap. This confirms
Exercise 1's numerical observation ($1.75=1.75$) is not a coincidence of that particular example,
but a general fact about dyadic distributions: Huffman coding is not just close to optimal for
them, it is *exactly* optimal, hitting the entropy bound on the nose.

</details>

### 🐍 IMPLEMENT
4. Implement `kraft_sum(codes)` that takes a codeword dictionary (like `assign_codewords`'s
   output) and returns $\sum_i 2^{-\ell_i}$. Run it on the Huffman codes produced by
   `build_huffman_tree` for many different randomly-generated frequency dictionaries (varying
   both the number of symbols and their frequencies). Confirm Kraft's inequality
   ($\sum_i 2^{-\ell_i}\le 1$) holds for every single one — and, specifically for Huffman's
   output (as opposed to prefix codes in general), check whether it holds with equality every
   time.

<details>
<summary>✅ How to know you're right</summary>

Across many random frequency dictionaries (try varying the alphabet size from 2 to 15+ symbols
and using widely different frequency skews), every single `kraft_sum` result for a Huffman-code
output should equal exactly `1.0` (not just $\le 1$) — Huffman's tree-construction guarantees
every internal node has exactly two children, so the resulting binary tree is *full* (no node has
only one child), and a full binary tree's leaves always satisfy Kraft's inequality with equality.
If you ever measure a sum strictly less than `1.0` for a genuine Huffman-tree output, that
indicates a bug in the tree construction (likely a node ending up with only one child) rather
than a real, valid-but-inefficient prefix code — Huffman specifically never produces those.

</details>

### 🤖 APPLY
5. A speech-to-text system's output vocabulary has highly skewed unigram frequencies (a few
   extremely common words, a long tail of rare ones) — model this with frequencies
   $\{\text{"the"}: 5000, \text{"a"}: 3000, \text{"is"}: 1500,\ldots\}$ down to several
   rare words with frequency 1, at least 15 words total. Build the Huffman code for this
   vocabulary, and compare the resulting average codeword length to (a) this notebook's entropy
   bound $H(X)$ and (b) a naive fixed-width code using $\lceil\log_2(\text{vocab size})\rceil$
   bits per word regardless of frequency. Report the compression ratio Huffman achieves over the
   fixed-width baseline.

<details>
<summary>✅ What you should observe</summary>

With a genuinely skewed frequency table (a realistic Zipf-like unigram distribution), the average
Huffman length should land close to $H(X)$ (within the guaranteed 1-bit gap, typically much
closer in practice for real skewed distributions), while the fixed-width baseline
($\lceil\log_2(n)\rceil$ bits for every word, ignoring frequency entirely) is noticeably
larger — for 15+ words, fixed-width needs at least 4 bits/word regardless of how common a word
is, while Huffman assigns the most frequent words (like "the") codewords of 1-3 bits. The
resulting compression ratio (fixed-width bits / Huffman bits) should exceed 1.0 by a comfortable
margin, growing larger the more skewed the frequency table is — directly reproducing this
notebook's own `longer_text` result (a $\sim$1.6x reduction there) but now on word-level rather
than character-level frequencies, the same principle BPE tokenizers exploit at scale.

</details>

### 🚀 CHALLENGE
6. Extend the `corpus` list in the AI section with several more words that share common
   substrings with `"lower"` and `"widest"` (e.g. add `"slow"`, `"slower"`, `"wildest"` a few
   times each), increasing their effective frequency. Re-run the BPE merges and check whether
   `"lower"` or `"widest"` now also collapse into single tokens. Connect your observation to why
   real tokenizer vocabularies are trained on massive corpora — rare combinations in a small toy
   corpus are common somewhere in a large enough one.

<details>
<summary>🔍 What a strong answer covers</summary>

A strong answer finds that adding `"slow"` (sharing "low" with the original `"low"`/`"lower"`)
and `"slower"` tips `"lower"` over into collapsing to a single token — the shared substring
`"low"` now accumulates enough combined frequency (from `"low"`, `"lower"`, *and* `"slow"`,
`"slower"` all containing it) to win early merges, and once `"lower"`'s full span has been merged
piece by piece across enough steps, it ends up as one token where it previously took two. Adding
`"wildest"` (sharing "wi" and "dest" with the original `"widest"`) has a smaller effect in a run
with modest added frequencies — `"widest"` may still take 2 tokens depending on exactly how much
frequency was added, illustrating that *which* words benefit from more corpus support is itself
frequency-dependent, not automatic. A strong answer connects this to why production tokenizers
(BPE for GPT models, WordPiece, SentencePiece) are trained on corpora with billions of words:
almost any substring pattern that looks "rare" in a small sample becomes common enough to merit
its own merge once the corpus is large enough — the toy corpus's limitations are a *sample size*
problem, not a flaw in the BPE algorithm itself.

</details>

---

## 📚 Further Reading
- Huffman, ["A Method for the Construction of Minimum-Redundancy Codes"](https://ieeexplore.ieee.org/document/4051119) (1952)
- Sennrich, Haddow & Birch, ["Neural Machine Translation of Rare Words with Subword Units"](https://arxiv.org/abs/1508.07909) (BPE for NLP)
- Cover & Thomas, *Elements of Information Theory*, Ch. 5 (Data Compression)

---

*Next notebook: [Information Theory in LLMs](07_information_theory_in_llms.ipynb)*